# Fine-tune LLM with Ray on SageMaker HyperPod (Studio Workflow)

**LoRA (optional FSDP) on a Ray cluster managed through SageMaker Studio — S3 variant (no FSx required)**

> This is a copy of `notebook.ipynb` that uses **Amazon S3** for every shared artifact (dataset, Ray Train
> checkpoints, final model) instead of an FSx for Lustre mount. It runs with `train_ray_s3.py`,
> `args-s3.yaml` and `requirements-s3.txt` from this folder.

This notebook uses the new **Ray on HyperPod** integration in SageMaker Studio. Studio handles
cluster creation, Ray Dashboard access, Amazon Managed Grafana observability, workspace
connectivity, and hung job detection — no kubectl, Helm, or Kubernetes YAML required.

### What You Do vs. What Studio Does

| You | Studio |
|-----|--------|
| Write training script (`train_ray.py`) | Creates Ray cluster via KubeRay operator |
| Configure hyperparameters (`args.yaml`) | Installs CRDs (RayCluster, RayJob, RayService) |
| Prepare dataset | Creates namespace and service account |
| Submit and monitor job | Provides Ray Dashboard with IAM-authenticated URLs |
| | Provisions 4 Grafana dashboards (Core, Data, Train, Serve) |
| | Connects JupyterLab/Code Editor as zero-compute worker |
| | Detects hung jobs via per-node Job Monitoring Agent |
| | Supports tiered checkpointing (local disk → S3) |

## Prerequisites

Before running this notebook, ensure four components are installed on your HyperPod EKS cluster:

| Component | Purpose |
|-----------|--------|
| **SageMaker Spaces EKS add-on** | Enables JupyterLab and Code Editor workspaces that attach to Ray clusters |
| **HyperPod Observability EKS add-on** | Collects metrics from Ray workloads, provisions Grafana dashboards |
| **KubeRay operator** | Manages RayCluster, RayJob, and RayService as native Kubernetes resources |
| **HyperPod Ray Endpoint Operator** (Helm chart) | Generates authenticated public endpoints for dashboard and remote job submission |

You also need:
- A **SageMaker Studio domain** with access to the HyperPod cluster
- An **S3 bucket/prefix** for the dataset, checkpoints, and model output (set `s3_base_uri` in `args-s3.yaml`)
- **EKS Pod Identity** associations that give the Ray head, Ray workers, and this space read/write access to
  that prefix. Ray pods on HyperPod cannot use the node's IMDS credentials, so without Pod Identity every S3
  call fails with `Unable to locate credentials` / `ACCESS_DENIED`. Associate the role with the `default`
  service account (space + workers) and the `<ray-cluster-name>` service account (head) in the Ray namespace.

FSx for Lustre is **not** required: scripts are shipped to the cluster with Ray's `runtime_env` `working_dir`,
and data moves through S3.

See the [Ray on HyperPod getting started guide](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-hyperpod-eks-ray.html) for full setup details.

## Step 1: Create a Ray Cluster from SageMaker Studio

1. Open **SageMaker Studio**
2. Navigate to **HyperPod** in the left sidebar
3. Select your HyperPod cluster
4. Go to the **Tasks** tab
5. From the task type list, choose **RayCluster**
6. Click **Create Ray Cluster** and configure:
   - **Cluster name**: `ray-medical-finetune`
   - **Head instance type**: select an instance (no GPU needed for head)
   - **Worker instance type**: a GPU instance with at least 8 spare vCPUs for one Ray Train worker
     (`train_ray_s3.py` requests 1 GPU + 8 CPUs per worker; tested on `g5.12xlarge` / `g6.12xlarge`, 4 GPUs each).
     A `g5.xlarge`/`g6.2xlarge` pod has fewer than 8 allocatable CPUs — lower `cpus_per_worker` in `args-s3.yaml` if you use one.
   - **Worker count**: 1 (scale up for larger models — see README)
   - **Container image**: use the default **SageMaker Distribution image** (Ray pre-installed, AWS-maintained with vulnerability patching)
7. Under advanced settings, **enable remote endpoints** for dashboard access and remote job submission
8. Click **Create**

Studio automatically:
- Registers CRDs (RayCluster, RayJob, RayService) via the KubeRay operator
- Creates the namespace and service account
- Deploys the RayCluster with your configuration
- Integrates with HyperPod task governance for compute quotas and scheduling priorities

Wait for the cluster status to show **Running** on the Tasks tab before continuing.

> **Tip**: For advanced customization, Studio provides an inline YAML editor that exposes the full
> Kubernetes manifest. Use this if you need custom tolerations, node selectors, or resource overrides.

> **Scaling**: The default config uses 1 worker for the 0.6B model. For larger models (7B+),
> increase the worker count and update `num_workers` in `args.yaml` to match.

## Step 2: Access the Ray Dashboard

Once the cluster is **Running**:

1. On the **Tasks** tab, find your Ray cluster
2. Click **Actions** → **Open Ray Dashboard**
3. Studio generates a **short-lived, IAM-authenticated URL** scoped to the cluster creator
4. The dashboard opens in a new browser tab showing cluster health, running jobs, and node status

No `kubectl port-forward` needed. The URL is secured via IAM and scoped to your identity.

## Step 3: Connect a Workspace to the Ray Cluster

1. In Studio, create a **HyperPod JupyterLab** or **Code Editor** space
2. During space creation, a **Ray cluster integration option** appears in the configuration panel
3. Select your Ray cluster (`ray-medical-finetune`)
4. The space restarts to attach to the cluster
5. Open the workspace in your browser

The workspace joins the cluster as a **zero-compute worker node**, giving you full native
Ray driver access. You work as if on the head node — `ray.init(address="auto")` connects
directly to the cluster.

**Open this notebook in the connected workspace to continue.**

## Step 4: Install Dependencies

In [1]:
# requirements-s3.txt deliberately does NOT pin Ray: the space and the Ray cluster must run the
# same Ray version as the cluster image. Installing another Ray version here breaks ray.init().
!pip install -q -r requirements-s3.txt
# pip may print dependency-conflict warnings for preinstalled image packages (autogluon, s3fs vs.
# fsspec). They don't affect this notebook: datasets needs fsspec<=2026.2.0 and S3 I/O uses boto3.


In [2]:
import os

# The SageMaker Distribution image includes TensorFlow + Keras 3, which transformers 4.x can't load
# ("Your currently installed version of Keras is Keras 3 ..."). This notebook is PyTorch-only,
# so disable the TF integration before anything imports transformers.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import ray

# Connect to the attached Ray cluster. The workspace joined as a zero-compute
# worker node during Step 3, so address="auto" finds the cluster automatically.
ray.init(address="auto")

print(f"Ray version: {ray.__version__}")
print(f"Available resources: {ray.cluster_resources()}")
print(f"Nodes: {len(ray.nodes())}")
ray.shutdown()

2026-10-08 13:10:15,198	INFO worker.py:1672 -- Using address auto set in the environment variable RAY_ADDRESS
2026-10-08 13:10:15,202	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: ray-cluster-1-head-svc.ray-hyperpod.svc.cluster.local:6379...
2026-10-08 13:10:15,213	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://10.3.185.120:8265 


Ray version: 2.55.1
Available resources: {'object_store_memory': 41407708773.0, 'memory': 197976154112.0, 'node:10.3.200.102': 1.0, 'CPU': 46.0, 'node:10.3.40.161': 1.0, 'GPU': 4.0, 'accelerator_type:L4': 1.0, 'node:__internal_head__': 1.0, 'node:10.3.185.120': 1.0}
Nodes: 3


### Install dependencies on Ray workers

Ray's `runtime_env` ships this notebook's folder (`working_dir`) and installs `requirements-s3.txt`
(`pip`) on the head and workers when the job starts — no image rebuild, no shared file system,
no `kubectl exec`. We use this when submitting the job in Step 7.

## Step 5: Prepare the Dataset

The dataset is written to **S3**. Each Ray Train worker downloads it to its local disk before training.
All paths derive from `s3_base_uri` in `args-s3.yaml`.

In [3]:
import os
import yaml
from datasets import load_dataset

with open("args-s3.yaml") as f:
    CONFIG = yaml.safe_load(f)

S3_BASE_URI = CONFIG["s3_base_uri"].rstrip("/")
MODEL_ID = CONFIG["script_args"]["model_id"]
DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES = 1000

print(f"S3 base URI: {S3_BASE_URI}")

dataset = load_dataset(DATASET_ID, "en", split=f"train[:{NUM_SAMPLES}]")
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:\n{dataset[0]}")

S3 base URI: s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical
Dataset loaded: 1000 samples
Columns: ['Question', 'Complex_CoT', 'Response']

Example:
{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?', 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen i

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = """You are a medical reasoning assistant. Respond to user queries
with clear, step-by-step medical reasoning."""


def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["Question"]},
        {
            "role": "assistant",
            "content": f"<think>\n{example['Complex_CoT']}\n</think>\n\n{example['Response']}",
        },
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example


dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 900, Val: 100


In [5]:
import boto3
from urllib.parse import urlparse

LOCAL_DATA = "/tmp/qwen3-medical/datasets"
os.makedirs(f"{LOCAL_DATA}/train", exist_ok=True)
os.makedirs(f"{LOCAL_DATA}/val", exist_ok=True)

# Keep only the formatted chat text — that's what SFTTrainer consumes (dataset_text_field="text").
train_dataset.select_columns(["text"]).to_json(f"{LOCAL_DATA}/train/train.json")
val_dataset.select_columns(["text"]).to_json(f"{LOCAL_DATA}/val/val.json")

_u = urlparse(S3_BASE_URI)
S3_BUCKET, S3_PREFIX = _u.netloc, _u.path.lstrip("/")
s3 = boto3.client("s3")
for split in ("train", "val"):
    key = f"{S3_PREFIX}/datasets/{split}/{split}.json"
    s3.upload_file(f"{LOCAL_DATA}/{split}/{split}.json", S3_BUCKET, key)
    print(f"Uploaded s3://{S3_BUCKET}/{key}")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploaded s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/datasets/train/train.json
Uploaded s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/datasets/val/val.json


## Step 6: Ship the Training Script with `runtime_env`

No copy step is needed. When the job is submitted, Ray zips this notebook's folder
(`working_dir`, minus the excludes below) and unpacks it on the head and on every worker.
`train_ray_s3.py` and `args-s3.yaml` are then available in the job's working directory.

In [6]:
WORKING_DIR = os.getcwd()
RUNTIME_EXCLUDES = [".git", "*.ipynb", ".ipynb_checkpoints", "__pycache__"]

with open("requirements-s3.txt") as f:
    PIP_PACKAGES = [l.strip() for l in f if l.strip() and not l.startswith("#")]

print(f"working_dir: {WORKING_DIR}")
print("Files shipped:", [f for f in os.listdir(WORKING_DIR) if f.endswith((".py", ".yaml", ".txt"))])
print("pip packages:", PIP_PACKAGES)

working_dir: /home/sagemaker-user/ray-hyperpod/Varun
Files shipped: ['args-s3.yaml', 'train_ray_s3.py', 'requirements-s3.txt']
pip packages: ['transformers==4.57.6', 'accelerate==1.14.0', 'datasets==4.8.5', 'peft==0.17.1', 'trl==0.24.0']


## Step 7: Submit the Training Job

Three options for job submission:

- **Option A**: Ray Jobs API from this notebook (programmatic)
- **Option B**: Remote CLI via `toolkit-for-ray-on-sagemaker-ai` (from laptop or CI/CD)
- **Option C**: `ray job submit` from the workspace terminal

### Resilience

The training script uses `FailureConfig(max_failures=3)`. If a node fails:
1. HyperPod automatically replaces the faulty node
2. Ray reschedules worker pods onto the new node
3. Training resumes from the last checkpoint (persisted to S3 via `RunConfig.storage_path`)

No changes to your training code are needed for this to work.

### Option A: Ray Jobs API (from this notebook)

In [7]:
from ray.job_submission import JobSubmissionClient

# The job client talks to the Ray dashboard HTTP API on the head service.
# Set RAY_CLUSTER_NAME / RAY_NAMESPACE if yours differ.
RAY_CLUSTER_NAME = os.environ.get("RAY_CLUSTER_NAME", "ray-cluster-1")
RAY_NAMESPACE = os.environ.get("RAY_NAMESPACE", "ray-hyperpod")
RAY_DASHBOARD = os.environ.get(
    "RAY_DASHBOARD_ADDRESS",
    f"http://{RAY_CLUSTER_NAME}-head-svc.{RAY_NAMESPACE}.svc.cluster.local:8265",
)

client = JobSubmissionClient(RAY_DASHBOARD)
print(f"Connected to Ray at: {RAY_DASHBOARD}")
print(f"Existing jobs: {len(client.list_jobs())}")

Connected to Ray at: http://ray-cluster-1-head-svc.ray-hyperpod.svc.cluster.local:8265
Existing jobs: 5


In [8]:
job_id = client.submit_job(
    entrypoint="python train_ray_s3.py --config args-s3.yaml",
    runtime_env={
        "working_dir": WORKING_DIR,
        "excludes": RUNTIME_EXCLUDES,
        "pip": PIP_PACKAGES,
        "env_vars": {
            # PyTorch-only job: stop transformers from importing the image's TF/Keras 3
            "USE_TF": "0",
            "TRANSFORMERS_NO_TF": "1",
            "NCCL_DEBUG": "WARN",
            "NCCL_SOCKET_IFNAME": "eth0",
        },
    },
    metadata={
        "name": "qwen3-medical-finetune",
        "cluster": "hyperpod-studio",
    },
)

print(f"Job submitted: {job_id}")
print(f"\nView in Ray Dashboard: Actions > Open Ray Dashboard in Studio")
print(f"View in Grafana:       Actions > Open Grafana in Studio")

2026-10-08 13:10:58,081	INFO dashboard_sdk.py:364 -- Uploading package gcs://_ray_pkg_96e427693a706770.zip.
2026-10-08 13:10:58,082	INFO packaging.py:691 -- Creating a file package for local module '/home/sagemaker-user/ray-hyperpod/Varun'.


Job submitted: raysubmit_Ta9JB3fTEvuTmmpC

View in Ray Dashboard: Actions > Open Ray Dashboard in Studio
View in Grafana:       Actions > Open Grafana in Studio


### Option B: Remote CLI (from laptop or CI/CD)

The `toolkit-for-ray-on-sagemaker-ai` package enables remote job submission from anywhere
with IAM credentials. It handles endpoint resolution and EKS API credential generation
through IAM authentication.

```bash
# One-time setup
pip install toolkit-for-ray-on-sagemaker-ai
aws eks update-kubeconfig --name <eks-cluster-name> --region <region>

# Submit job using the sagemaker_ray:// address scheme
# Run from this folder (it is uploaded as the working dir)
ray job submit \
    --address sagemaker_ray://<ray-cluster-name>/<namespace> \
    --working-dir . \
    --runtime-env-json '{"pip": "requirements-s3.txt"}' \
    -- python train_ray_s3.py --config args-s3.yaml

# List jobs
ray job list --address sagemaker_ray://<ray-cluster-name>/<namespace>
```

Replace `<ray-cluster-name>` and `<namespace>` with your cluster's values from the Studio Tasks tab.

### Option C: Workspace terminal

Open a terminal in this JupyterLab workspace, `cd` into this folder, and run:

```bash
export RAY_ADDRESS=http://ray-cluster-1-head-svc.ray-hyperpod.svc.cluster.local:8265
ray job submit \
    --working-dir . \
    --runtime-env-json '{"pip": "requirements-s3.txt", "excludes": [".git", "*.ipynb"], "env_vars": {"USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "NCCL_DEBUG": "WARN"}}' \
    -- python train_ray_s3.py --config args-s3.yaml
```

## Step 8: Monitor the Job

### From Studio (recommended)

Three monitoring surfaces, all accessible from the **Tasks tab Actions menu**:

1. **Ray Dashboard** — Click **Open Ray Dashboard**. Shows job status, worker logs,
   resource utilization, and task timeline. Uses IAM-authenticated short-lived URLs.

2. **Amazon Managed Grafana** — Click **Open Grafana**. The HyperPod Observability add-on
   auto-discovers Ray pods, scrapes their metrics endpoints, and provisions **four pre-built
   dashboards** under a Ray folder:
   - **Ray Core** — cluster-level resource utilization
   - **Ray Data** — data loading and preprocessing metrics
   - **Ray Train** — training throughput, loss curves, GPU utilization
   - **Ray Serve** — inference endpoint metrics

   Dashboards support filtering by specific Ray cluster and appear alongside existing
   HyperPod infrastructure dashboards (GPU, EFA, task governance).

3. **Hung job detection** — The per-node **Job Monitoring Agent** monitors node-level and
   job-level signals to detect when training has stalled (e.g., a pod fails and other pods
   block at the next collective operation). No code changes required for Ray Train workloads.
   Notifications are sent to:
   - The cluster's **Amazon CloudWatch log group**
   - The **Ray Train Grafana dashboard**

### From this notebook (programmatic)

In [9]:
import time

while True:
    status = client.get_job_status(job_id)
    print(f"Status: {status}")
    if status.is_terminal():
        break
    time.sleep(30)

logs = client.get_job_logs(job_id)
print(f"\n{'=' * 60}")
print(f"Job Logs (last 2000 chars):")
print(f"{'=' * 60}")
print(logs[-2000:])

Status: SUCCEEDED

Job Logs (last 2000 chars):
 into base model...
(RayTrainWorker pid=33527, ip=10.3.40.161) Uploaded merged model (10 files) to s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/output/merged_model
(RayTrainWorker pid=33527, ip=10.3.40.161) Checkpoint successfully created at: Checkpoint(filesystem=s3, path=sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/ray_results/hyperpod-ray-finetune/checkpoint_2026-10-08_13-19-15.216899)
(RayTrainWorker pid=33527, ip=10.3.40.161) Reporting training result 12: TrainingReport(checkpoint=Checkpoint(filesystem=s3, path=sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/ray_results/hyperpod-ray-finetune/checkpoint_2026-10-08_13-19-15.216899), metrics={'train_runtime': 463.7803, 'train_samples_per_second': 3.881, 'train_steps_per_second': 0.97, 'total_flos': 3521245362520064.0, 'train_loss': 1.6692589908175999, 'epoch': 2.0, 'eval_loss': 1.7029750347137451, 'eval_runtime': 7.6668, 'eval

## Step 9: Configure Hung Job Detection (Optional)

The built-in hung job detection works out of the box for Ray Train workloads. For custom
detection rules (specific log patterns or timeout thresholds), use the
`toolkit-for-ray-on-sagemaker-ai` library.

When the configured action is `cancel`, HyperPod terminates the hung worker process, and
Ray Train's `FailureConfig` restarts workers from the last checkpoint automatically.

In [ ]:
# Example: custom hung job detection rules (optional)
# The API below is illustrative — refer to the toolkit-for-ray-on-sagemaker-ai
# documentation for the exact module and method signatures.

# from toolkit_for_ray_on_sagemaker_ai import hung_job_detection
#
# hung_job_detection.configure(
#     cluster_name="ray-medical-finetune",
#     rules=[
#         {
#             "name": "no-gpu-activity",
#             "log_pattern": "GPU utilization: 0%",
#             "timeout_seconds": 600,
#             "action": "cancel",
#         },
#     ],
# )

## Step 10: Verify Output

Model artifacts are in S3 under `<s3_base_uri>/output/`. Ray Train checkpoints are under `<s3_base_uri>/ray_results/`.

In [10]:
def list_s3(prefix):
    total, files = 0, 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            total += obj["Size"]; files += 1
    return files, total

for sub in ("output/final_model", "output/merged_model", "ray_results"):
    n, size = list_s3(f"{S3_PREFIX}/{sub}/")
    status = f"{n} files, {size / 1e9:.2f} GB" if n else "not found — job may still be running"
    print(f"  s3://{S3_BUCKET}/{S3_PREFIX}/{sub}/  ->  {status}")

  s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/output/final_model/  ->  11 files, 0.06 GB
  s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/output/merged_model/  ->  10 files, 1.21 GB
  s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/ray_results/  ->  48 files, 0.33 GB


In [11]:
from transformers import pipeline

MERGED_S3_PREFIX = f"{S3_PREFIX}/output/merged_model/"
MERGED_PATH = "/tmp/qwen3-medical/merged_model"

n, _ = list_s3(MERGED_S3_PREFIX)
if n:
    print(f"Downloading merged model ({n} files) to {MERGED_PATH} ...")
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=MERGED_S3_PREFIX):
        for obj in page.get("Contents", []):
            dest = os.path.join(MERGED_PATH, obj["Key"][len(MERGED_S3_PREFIX):])
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            s3.download_file(S3_BUCKET, obj["Key"], dest)

    # The space has no GPU; a 0.6B model generates on CPU in about a minute.
    pipe = pipeline("text-generation", model=MERGED_PATH, torch_dtype="auto", device_map="auto")

    test_prompt = "What are the common symptoms and diagnostic criteria for Type 2 Diabetes?"
    result = pipe([{"role": "user", "content": test_prompt}], max_new_tokens=256)
    print(f"\nPrompt: {test_prompt}")
    print(f"\nResponse:\n{result[0]['generated_text'][-1]['content']}")
else:
    print(f"Merged model not found at s3://{S3_BUCKET}/{MERGED_S3_PREFIX}")
    print("Check that merge_weights is enabled in args-s3.yaml and the job completed.")

`torch_dtype` is deprecated! Use `dtype` instead!
The tokenizer you are loading from '/tmp/qwen3-medical/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cpu



Prompt: What are the common symptoms and diagnostic criteria for Type 2 Diabetes?

Response:
<think>
Alright, so Type 2 Diabetes is all about how our body can't make enough insulin to keep things stable. It's like a system problem where the pancreas doesn't have enough insulin to handle the extra glucose that the body needs. Hmm, let's think about the symptoms. When I see someone with Type 2 Diabetes, the first thing that rings a bell is fatigue and weakness. That's usually because the blood sugar levels are really high and not getting processed properly.

Oh, then there's blurry vision. This might happen because the blood sugar is too high and affecting the eye, leading to blurry vision. And then, there's dry mouth and constipation. These are the body's way of saying it's not getting enough glucose, so it's trying to hold onto it. Plus, weight gain can pop into the picture, where people gain extra pounds even though they are still eating the same amount. It's all part of the body's e

## Step 11: Model Location in S3

The training job already uploaded the merged model to S3, so no extra upload step is needed.
Use this URI for deployment (SageMaker endpoints, Bedrock Custom Model Import, etc.), or copy it to a
dedicated model bucket.

In [12]:
print(f"Merged model:  s3://{S3_BUCKET}/{S3_PREFIX}/output/merged_model/")
print(f"LoRA adapter:  s3://{S3_BUCKET}/{S3_PREFIX}/output/final_model/")

Merged model:  s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/output/merged_model/
LoRA adapter:  s3://sagemaker-ray-hyperpod-f54c7436-bucket/ray-results/qwen3-medical/output/final_model/


## Step 12: Deploy with Ray Serve

Deploy the fine-tuned model as a live HTTP endpoint using [Ray Serve](https://docs.ray.io/en/latest/serve/index.html).
The deployment runs on the same Ray cluster — no additional infrastructure needed.

The Serve application:
1. Downloads the merged model from S3 to the worker's local disk
2. Loads it on GPU with the HuggingFace `pipeline` API
3. Serves HTTP requests at the Ray head node on port 8000

> **Alternative**: Studio also supports creating a **RayService** from the Tasks tab for a
> fully managed Serve deployment with health checks, automatic rollouts, and its own
> IAM-authenticated endpoint.

In [ ]:
import ray
from ray import serve

ray.init(address="auto", ignore_reinit_error=True)

SERVE_MODEL_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/output/merged_model"

@serve.deployment(
    ray_actor_options={"num_gpus": 1, "num_cpus": 2},
    num_replicas=1,
)
class MedicalQA:
    def __init__(self, model_s3_uri: str):
        import os, boto3, torch
        from urllib.parse import urlparse
        from transformers import pipeline as hf_pipeline

        local_path = "/tmp/qwen3-medical/serve_model"
        parsed = urlparse(model_s3_uri)
        bucket = parsed.netloc
        prefix = parsed.path.lstrip("/").rstrip("/") + "/"
        s3_client = boto3.client("s3")
        for page in s3_client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                rel = obj["Key"][len(prefix):]
                if not rel or rel.endswith("/"):
                    continue
                dest = os.path.join(local_path, rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                s3_client.download_file(bucket, obj["Key"], dest)
        print(f"Model downloaded from {model_s3_uri}")

        self.pipe = hf_pipeline(
            "text-generation",
            model=local_path,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        print("Model loaded — ready for inference")

    async def __call__(self, http_request):
        data = await http_request.json()
        prompt = data.get("prompt", "")
        max_tokens = data.get("max_tokens", 256)
        messages = [{"role": "user", "content": prompt}]
        result = self.pipe(messages, max_new_tokens=max_tokens, do_sample=True, temperature=0.7)
        return {"response": result[0]["generated_text"][-1]["content"]}


app = MedicalQA.bind(model_s3_uri=SERVE_MODEL_URI)
handle = serve.run(app, name="medical-qa", route_prefix="/medical-qa")

SERVE_URL = f"http://{RAY_CLUSTER_NAME}-head-svc.{RAY_NAMESPACE}.svc.cluster.local:8000/medical-qa"
print(f"Deployment live at: {SERVE_URL}")
print("View in Ray Dashboard → Serve tab for deployment status, replicas, and request metrics")

In [ ]:
import requests

test_prompts = [
    "What are the common symptoms and diagnostic criteria for Type 2 Diabetes?",
    "Explain the mechanism of action of metformin for blood sugar control.",
]

for prompt in test_prompts:
    resp = requests.post(SERVE_URL, json={"prompt": prompt, "max_tokens": 200})
    result = resp.json()
    print(f"Prompt: {prompt}\n")
    print(f"Response: {result['response'][:500]}\n")
    print("-" * 60)

### Accessing the endpoint

| From | How |
|------|-----|
| **This workspace** | Use the in-cluster URL above (`http://<head-svc>:8000/medical-qa`) |
| **Another pod in the cluster** | Same URL — any pod can reach the head service via Kubernetes DNS |
| **Outside the cluster** | Use the IAM-authenticated endpoint from Studio (**Tasks tab → Actions**), or `kubectl port-forward svc/<cluster>-head-svc 8000:8000` |

### Scaling the deployment

- **Horizontal**: increase `num_replicas` in `@serve.deployment` to spread load across GPU workers
- **Model parallelism**: for models too large for one GPU, set `ray_actor_options={"num_gpus": N}` and use `device_map="auto"` to shard across GPUs within one replica
- Ray Serve handles load balancing and health checks automatically
- The **Ray Serve Grafana dashboard** (provisioned by the Observability add-on) shows request latency, throughput, and replica status

In [ ]:
# Shut down the Serve application when done (uncomment to execute)
# serve.delete("medical-qa")
# ray.shutdown()
# print("Serve deployment 'medical-qa' deleted.")

print("To shut down: serve.delete('medical-qa')")
print("The deployment persists until deleted or the Ray cluster is terminated.")

## Step 13: Scale and Iterate

One of the key advantages of this workflow: you can scale without recreating the cluster.

- **Scale workers** from Studio: edit the cluster and change the worker count
- **Scale from code**: change `num_workers` in `args.yaml` — prototype with 1 worker, then
  scale to 8+ by changing one line in `ScalingConfig`
- **Scale Serve replicas**: increase `num_replicas` in the deployment to handle more inference traffic
- **The notebook stays interactive** while training runs distributed across the cluster —
  monitor, adjust hyperparameters, or inspect intermediate results

## Step 14: Clean Up

Manage the Ray cluster from Studio:

1. On Studio's **Tasks** tab, select your Ray cluster
2. Click **Actions** → **Delete**

HyperPod nodes remain available for other workloads after the Ray cluster is deleted.
To stop incurring compute charges, scale the HyperPod instance groups to 0 or delete the HyperPod cluster.

In [ ]:
# Optionally delete the dataset, checkpoints, and model artifacts from S3

# for page in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=f"{S3_PREFIX}/"):
#     objs = [{"Key": o["Key"]} for o in page.get("Contents", [])]
#     if objs:
#         s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": objs})
# print(f"Deleted s3://{S3_BUCKET}/{S3_PREFIX}/")

print("Cleanup commands above are commented out for safety. Uncomment to execute.")

## Summary

| Step | What happened |
|------|---------------|
| 1 | Created Ray cluster from SageMaker Studio Tasks tab |
| 2 | Opened Ray Dashboard via IAM-authenticated URL |
| 3 | Connected JupyterLab workspace (zero-compute worker node) |
| 4 | Installed Python dependencies |
| 5 | Prepared dataset and uploaded it to S3 |
| 6 | Shipped the training script with `runtime_env` `working_dir` |
| 7 | Submitted Ray training job (3 submission options) |
| 8 | Monitored via Ray Dashboard, Managed Grafana, and hung job detection |
| 9 | (Optional) Configured custom hung job detection rules |
| 10 | Verified model output in S3 and ran a test prompt |
| 11 | Located the model in S3 for deployment |
| 12 | Deployed the model as an HTTP endpoint with Ray Serve |
| 13 | Scaled workers and Serve replicas without recreating the cluster |
| 14 | Cleaned up Ray cluster from Studio |

### What We Didn't Have to Do

- Install or configure `kubectl` or `helm`
- Write Kubernetes YAML manifests (RayCluster, Namespace, ServiceAccount)
- Provision or mount a shared file system (FSx)
- Install CRDs (KubeRay operator, Kubeflow Training Operator)
- Debug pod scheduling, node capacity, or webhook failures
- Set up `kubectl port-forward` for the Ray Dashboard
- Configure Prometheus and Grafana for observability
- Rebuild Docker images for dependency changes
- Use `kubectl cp` or `kubectl exec` to move files and install packages
- Configure PodMonitors, ServiceMonitors, or scrape targets
- Set up IAM roles for SigV4 signing
- Set up a separate inference server (vLLM, TGI) or load balancer

### Resilience Features (automatic)

| Layer | What it does |
|-------|-------------|
| **Node recovery** | HyperPod replaces faulty nodes; Ray reschedules pods; `FailureConfig` retries the job |
| **Hung job detection** | Per-node Job Monitoring Agent detects stalls; alerts via CloudWatch and Grafana |
| **Tiered checkpointing** | `amzn-sagemaker-checkpointing` writes to local disk first, async uploads to S3 |
| **Serve health checks** | Ray Serve monitors replica health; unhealthy replicas are restarted automatically |

### Key Packages

| Package | Purpose |
|---------|--------|
| `toolkit-for-ray-on-sagemaker-ai` | Remote job submission (`sagemaker_ray://` address), JumpStart model loader, custom hung job detection rules |
| `amzn-sagemaker-checkpointing` | Tiered checkpointing integration (local disk → S3) |
| SageMaker Distribution image | Default container with Ray pre-installed (AWS-maintained) |